# Поисковая лаборатория: пять fulltext и пять embedding механизмов

Основной цикл: query и явные hard filters → preview → compare → выдачи и матрица → полные карточки.
Меняйте одну входную ячейку и повторяйте запуск. Чтение snapshot не делает сети. Разметка не требуется.

Первый запуск:
1. Откройте notebook в JupyterLab/VS Code с Python kernel окружения проекта. Jupyter предоставляет оператор.
2. Скопируйте `examples/study/site.example.yaml` в свой site YAML. Заполните OpenSearch index/fields, URL и scalar bindings. Для dense нужны pinned TEI и ready whole/title/body/chunk generations общего cohort. Credentials задавайте через environment; не вставляйте их в ячейки.
3. Укажите SITE_CONFIG и OVERRIDES ниже. Можно передать RECSYS_STUDY_SITE и RECSYS_STUDY_OVERRIDES (JSON array Hydra overrides) в environment kernel.
4. Выполните setup, входную ячейку, preview и compare. Успешный полный поиск выбранного набора означает готовность исследования. Недоступный вариант даёт ошибку, без fallback.

[Спека рабочего места](../../../docs/specs/009-search-experiment-matrix.md), [подготовка поколений](../../../docs/specs/008-vector-search.md), [README](../../../README.md).
Instantiate не проверяет remote readiness. Обязательный актуальный admission выбранных dense generations выполняется перед поиском. OpenSearch proxy используется только для search.
Run All может выполнить демонстрационный compare; save, audit, batch и build выключены по умолчанию.


In [ ]:
import json
import os
import sys
from pathlib import Path

from IPython.display import HTML, display  # ty: ignore[unresolved-import]

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/main.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SITE_CONFIG = Path(os.environ.get("RECSYS_STUDY_SITE", "/srv/recsys/site.yaml"))
OVERRIDES = json.loads(os.environ.get("RECSYS_STUDY_OVERRIDES", "[]"))

In [ ]:
from src.experiments.research.notebook import (
    StudyNotebookSession,
    compose_site_config,
    compose_site_experiment,
)
from src.experiments.research.study import StudyExperiment

# Локальная настройка: без search, inference, build и export.
# После изменения YAML/OVERRIDES повторите compose: объекты не являются live-view.
study = compose_site_experiment(SITE_CONFIG, OVERRIDES, config_root=ROOT / "configs")
assert isinstance(study, StudyExperiment)
session = StudyNotebookSession(study)
display(
    HTML(
        "<p>Настроено; удалённые prerequisites ещё не проверены. Коллекции этой ячейкой не строятся.</p>"
    )
)
display(study.describe())

## Входы одного опыта

Fulltext: `["f1", "f2", "f3", "f4", "f5"]`; embedding: `["e1", "e2", "e3", "e4", "e5"]`; all ten — оба списка.
Variants — непустой поднабор зарегистрированных вариантов без дубликатов. Reference входит в него и меняет только отображение.
Hard filters задаются отдельно от мягких пожеланий query. False — действующий boolean filter. Static filters site остаются AND; новая per-call city их не отменяет.
Query/city/variants меняются без build. Depth, α, RRF, exact/ANN и bindings требуют новой compose/instantiate. Document recipe/model/cohort требуют нового поколения.


In [ ]:
text = "комплектовщик ТСД без опыта 2/2"
variants = ["f2", "f3", "f4", "e1", "e2"]
filter_values = {"city": "Москва", "active": True}
reference_variant = "f2"

## Preview: без сети

Проверьте effective filters, thematic core, soft spans, prepared inputs и declared abstention. План не обещает фактический порядок документов.


In [ ]:
preview = session.preview(text, variants=variants, filter_values=filter_values)
display(preview)

## Compare: внешние read-only вызовы

Каждый запуск делает новый поиск. Current сбрасывается до проверки входов; при ошибке чтение/save текущего результата недоступны. После исправления повторите запуск в том же kernel.
Старые Jupyter outputs могут остаться на экране; каждый результат подписан своим run_id/query/filters.


In [ ]:
session.compare(
    text,
    variants=variants,
    filter_values=filter_values,
    reference_variant=reference_variant,
)

## Паспорт, выдачи и матрица: без сети

Display subset и reference меняются без compare. Delta считается по рангам; score разных типов не вычитается. При несопоставимых источниках матрицы разделены; одинаковые строковые ids не доказывают общую вакансию.


In [ ]:
snapshot = session.require_current()
display(HTML(snapshot.summary_html()))

In [ ]:
display_variants = None  # Например ["f2", "f3"], только из этого snapshot.
display_reference = None  # None сохраняет reference snapshot.
snapshot = session.require_current()
display(HTML(snapshot.rank_lists_html(variants=display_variants)))

In [ ]:
snapshot = session.require_current()
display(
    HTML(
        snapshot.matrix_html(
            variants=display_variants, reference_variant=display_reference
        )
    )
)

## Полная карточка: без fetch

Задайте точный document_id из выдачи и при необходимости card_variant для различения snapshots. None открывает первую доступную карточку.
Excerpt в таблицах не сокращает полный description. Неизвестный id — локальная ошибка. Source HTML экранируется; missing условия показываются «не указано».


In [ ]:
document_id = None  # Вставьте точный id интересующей вакансии.
card_variant = None
snapshot = session.require_current()
if document_id is None:
    document_id = next(
        (item.id for _, _, _, result in snapshot.results() for item in result.items),
        None,
    )
if document_id is None:
    display(
        HTML("<p>Все выбранные выдачи пусты: 0 результатов. Карточка недоступна.</p>")
    )
else:
    display(HTML(snapshot.card_html(document_id, variant=card_variant)))

In [ ]:
# Данные берутся только из artifact; field/term explanations не выдумываются.
snapshot = session.require_current()
display(HTML(snapshot.diagnostics_html()))

## Следующий опыт и явное сохранение

1. Уберите «без опыта 2/2» из query. Повторите входную ячейку, preview/compare и чтение; сравните active paths.
2. Верните query и замените city на Санкт-Петербург. Per-call filters не накапливаются.
3. Для α-controls добавьте `study.preference_budget=0.1` в OVERRIDES и повторите compose/compare. Совместимые поколения не строятся заново.
4. Чтобы сохранить выбранный текущий snapshot, включите SAVE_CURRENT. Save не делает search и не использует изменённые входы для подмены старого результата.

Перед следующим compare можно присвоить успешный snapshot отдельной Python-переменной. Он сохраняет прежние run_id/query/filters и не становится current после неудачи нового опыта.
Файл `outputs/search-study/<run_id>/search-study.json` содержит входы, выдачи, provenance и identities, без credentials. Повторная запись того же пути запрещена. Replay/kernel restore не входят в сценарий.


In [ ]:
SAVE_CURRENT = False
OUTPUT_ROOT = ROOT / "outputs/search-study"
if SAVE_CURRENT:
    saved_path = session.save(OUTPUT_ROOT)
    display(str(saved_path))

## Дополнительные действия: только явный opt-in

Все flags False. Обычный Run All не читает весь корпус, не токенизирует его, не строит поколения, не запускает CSV и не публикует файлы.
Launcher/model download отсутствуют: инфраструктуру готовит оператор на целевой машине. Build пишет в Qdrant и требует нового физического имени; готовые поколения не перезаписываются.


In [ ]:
RUN_CORPUS_AUDIT = False
if RUN_CORPUS_AUDIT:
    from hydra.utils import instantiate

    configuration = compose_site_config(
        SITE_CONFIG, OVERRIDES, config_root=ROOT / "configs"
    )
    reader = instantiate(configuration.corpus.reader, _convert_="all")
    display(reader.audit())

In [ ]:
RUN_TOKEN_AUDIT = False
if RUN_TOKEN_AUDIT:
    from hydra.utils import instantiate

    configuration = compose_site_config(
        SITE_CONFIG, OVERRIDES, config_root=ROOT / "configs"
    )
    encoder = instantiate(configuration.embedding.encoder, _convert_="all")
    actual_input = encoder.prepare_query(text)
    display(encoder.integration.tokenize((actual_input,)))

In [ ]:
RUN_NATIVE_EMBEDDING = False
if RUN_NATIVE_EMBEDDING:
    from hydra.utils import instantiate

    from src.domain.serialization import fingerprint

    configuration = compose_site_config(
        SITE_CONFIG, OVERRIDES, config_root=ROOT / "configs"
    )
    encoder = instantiate(configuration.embedding.encoder, _convert_="all")
    actual_embedding = encoder.encode_query(text)
    display(
        {
            "model_input": encoder.prepare_query(text),
            "profile_id": actual_embedding.profile_id,
            "dimensions": len(actual_embedding.values),
            "vector_sha256": fingerprint(list(actual_embedding.values)),
        }
    )

In [ ]:
RUN_BATCH = False
BATCH_PATH = ROOT / "examples/study/queries.csv"
if RUN_BATCH:
    from src.experiments.batch.search import read_queries

    batch_snapshot = study.compare_queries(
        read_queries(BATCH_PATH),
        variants=variants,
        filters=study.bindings.terms(filter_values),
        reference_variant=reference_variant,
    )
    display(HTML(batch_snapshot._repr_html_()))

In [ ]:
RUN_BUILD = False
BUILD_KIND = "whole"  # whole / title / body / chunk
FRESH_COLLECTION = None
if RUN_BUILD:
    if BUILD_KIND not in {"whole", "title", "body", "chunk"} or not FRESH_COLLECTION:
        raise ValueError("Укажите build kind и новое физическое имя FRESH_COLLECTION")
    build_overrides = [
        *OVERRIDES,
        f"study.build_kind={BUILD_KIND}",
        f"study.collections.{BUILD_KIND}={FRESH_COLLECTION}",
    ]
    builder = compose_site_experiment(
        SITE_CONFIG,
        build_overrides,
        selection="chunk-index" if BUILD_KIND == "chunk" else "index",
        config_root=ROOT / "configs",
    )
    build_receipt = (
        builder.run()
    )  # Remote writes; exporters отключены, файлы не публикуются.
    display(build_receipt.to_dict())